In [1]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer,StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
def get_data(filename):

    return pd.read_csv(os.path.join('..','data','processed',f'{filename}.csv'))

In [3]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # Avoid division by zero in MAPE
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    print("----- Model Evaluation -----")
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"MAPE : {mape:.2f}%")

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "MAPE": mape
    }

In [4]:
training_data = get_data('training_data')
testing_data = get_data('testing_data')

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_numeric_distributions(df, cols, log_transform=False):
    n_cols = 2
    n_rows = len(cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))

    # If only one feature, fix axes shape
    if n_rows == 1:
        axes = np.array([axes])

    for i, col in enumerate(cols):
        data = df[col].dropna()

        # Optional log transform (for skewed data)
        if log_transform:
            data = np.log1p(data)

        # Histogram with KDE
        sns.histplot(data, kde=True, ax=axes[i, 0])
        axes[i, 0].set_title(f'{col} Distribution')

        # Boxplot
        sns.boxplot(y=data, ax=axes[i, 1])
        axes[i, 1].set_title(f'{col} Boxplot')

    plt.tight_layout()
    plt.show()

In [7]:
num_cols = [
    # Core logistics
    'delivery_distance',
    'total_product_weight_g',
    'total_volume',
    'freight_ratio',
    'weight_per_item',
    'volume_per_item',

    # Order info
    'n_items',
    'total_price',
    'payment_value',
    'payment_installments',

    # Interactions
    'distance_x_weight',
    'logistics_complexity',

    # Seller behavior
    'seller_previous_order_count',
    'seller_timely_delivery_avg',

    # Structural
    'n_unique_sellers',
    

]

obj_cols = ['category','payment_type']
binary_cols = ['is_same_state']

In [8]:
delivery_days_upper_limit =  training_data['delivery_days'].quantile(0.99)
traing_data_clip  = training_data[training_data['delivery_days']<=delivery_days_upper_limit]
testing_data_clip  = testing_data[testing_data['delivery_days']<=delivery_days_upper_limit]

In [9]:
x_train = traing_data_clip[num_cols+obj_cols+binary_cols]
y_train = traing_data_clip['delivery_days']
y_train_transformed = np.log1p(y_train)

x_test = testing_data_clip[num_cols + obj_cols +binary_cols]
y_test = testing_data_clip['delivery_days']
y_test_transformed = np.log1p(y_test)

In [10]:
x_train[num_cols].head()

,delivery_distance,total_product_weight_g,total_volume,freight_ratio,weight_per_item,volume_per_item,n_items,total_price,payment_value,payment_installments,distance_x_weight,logistics_complexity,seller_previous_order_count,seller_timely_delivery_avg,n_unique_sellers
0,286.0,200.0,2016.0,0.321090,100.0,1008.0,1.0,55.9,74.17,1.0,57200.0,1.0,26,0.961538,1.0
1,520.0,19100.0,104490.0,0.116827,9550.0,52245.0,1.0,165.4,184.84,1.0,9932000.0,1.0,7,0.285714,1.0
2,305.0,1450.0,10000.0,0.536948,725.0,5000.0,1.0,23.9,37.27,1.0,442250.0,1.0,34,0.823529,1.0
3,183.0,5750.0,27000.0,0.225100,2875.0,13500.0,1.0,99.0,121.51,1.0,1052250.0,1.0,0,NaN,1.0
4,806.0,100.0,3328.0,0.072369,50.0,1664.0,1.0,229.9,246.61,1.0,80600.0,1.0,64,1.000000,1.0


In [11]:
x_train[num_cols].describe()

,delivery_distance,total_product_weight_g,total_volume,freight_ratio,weight_per_item,volume_per_item,n_items,total_price,payment_value,payment_installments,distance_x_weight,logistics_complexity,seller_previous_order_count,seller_timely_delivery_avg,n_unique_sellers
count,76069.000000,76451.000000,7.645100e+04,76451.000000,76451.000000,76451.000000,76451.000000,76451.000000,76451.000000,76451.000000,7.606900e+04,76451.000000,76451.000000,74132.000000,76451.000000
mean,596.785313,2381.603694,1.732846e+04,0.298991,1085.962727,7871.157460,1.141424,136.660821,156.679325,2.913461,1.398430e+06,1.026278,182.468588,0.908017,1.014388
std,590.537383,4777.066149,3.028057e+04,0.280823,1947.282894,12125.645701,0.531037,208.713770,215.806811,2.703691,4.155162e+06,0.266465,295.033819,0.146278,0.126231
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,1.000000,0.850000,0.010000,0.000000,0.000000e+00,1.000000,0.000000,0.000000,1.000000
25%,180.000000,300.000000,2.964000e+03,0.130811,150.000000,1440.000000,1.000000,45.900000,59.895000,1.000000,8.800000e+04,1.000000,15.000000,0.884615,1.000000
50%,431.000000,750.000000,7.280000e+03,0.221439,350.000000,3375.000000,1.000000,86.000000,103.110000,2.000000,3.097920e+05,1.000000,59.000000,0.960000,1.000000
75%,794.000000,2050.000000,1.987200e+04,0.372249,933.500000,9600.000000,1.000000,149.900000,174.085000,4.000000,1.038800e+06,1.000000,198.000000,1.000000,1.000000
max,8678.000000,184400.000000,1.476000e+06,9.854054,22500.000000,210857.142857,21.000000,13440.000000,13664.080000,24.000000,2.394348e+08,15.000000,1808.000000,1.000000,5.000000


In [12]:
training_data['delivery_distance'].median()

np.float64(434.0)

In [15]:
# removing the outliers.
training_data =  training_data[training_data['delivery_distance']<4000]
testing_data = testing_data[testing_data['delivery_distance']<4000]

In [ ]:
plot_numeric_distributions(x_train,num_cols)

In [17]:
for col in obj_cols:
    print(x_train[col].value_counts().sort_values(ascending=False)[:10],end=f'\n{'_'*40}')
    

category
bed_bath_table           7216
health_beauty            6846
sports_leisure           5892
computers_accessories    5140
furniture_decor          4919
housewares               4536
watches_gifts            4307
telephony                3259
auto                     2990
toys                     2988
Name: count, dtype: int64
________________________________________payment_type
credit_card    58185
boleto         15003
voucher         2078
debit_card      1185
Name: count, dtype: int64
________________________________________

In [18]:
top_cols_value =  x_train['category'].value_counts().sort_values(ascending=False).nlargest(10).index.tolist()

x_train.loc[:,'category_topfeatures'] = np.where(x_train['category'].isin(top_cols_value), x_train['category'],'other')

x_test.loc[:,'category_topfeatures'] = np.where(x_test['category'].isin(top_cols_value), x_test['category'],'other')

C:\Users\HP\AppData\Local\Temp\ipykernel_25760\3128092168.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_train.loc[:,'category_topfeatures'] = np.where(x_train['category'].isin(top_cols_value), x_train['category'],'other')
C:\Users\HP\AppData\Local\Temp\ipykernel_25760\3128092168.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_test.loc[:,'category_topfeatures'] = np.where(x_test['category'].isin(top_cols_value), x_test['category'],'other')


In [19]:
skewed_cols = [
    # Core logistics
    'delivery_distance',
    'total_product_weight_g',
    'total_volume',
    'freight_ratio',
    'weight_per_item',
    'volume_per_item',

    # Order info
    'n_items',
    'total_price',
    'payment_value',
    'payment_installments',

    # Interactions
    'distance_x_weight',
    'logistics_complexity',

    # Seller behavior
    'seller_previous_order_count',
    'seller_timely_delivery_avg',

    # Structural
    'n_unique_sellers',
    

]
normal_cols = []

categorical_cols = ['category_topfeatures','payment_type']
binary_cols = ['is_same_state']

### Model Building

In [20]:
skwed_num_pipeline = Pipeline(steps=[
    ('imputation',SimpleImputer(strategy='median')),
    ('log',PowerTransformer(method='yeo-johnson')),
    ('standardize',StandardScaler())
] )

normal_num_pipeline = Pipeline(steps=[
    ('imputation',SimpleImputer(strategy='median')),
    ('standardize',StandardScaler())
] )

object_pipeline = Pipeline(
    steps = [
    
    ('impute',SimpleImputer(strategy='most_frequent')),
        
     ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
binary_cols_pipeline = Pipeline(steps=[
    ('imputation',SimpleImputer(strategy='most_frequent')),
   
] )


In [21]:

preprocessor = ColumnTransformer(transformers=[
    ('skewed', skwed_num_pipeline, skewed_cols),
    ('normal', normal_num_pipeline, normal_cols),
    ('categorical', object_pipeline, categorical_cols),
    ('binary', binary_cols_pipeline, binary_cols)
])

In [22]:
from sklearn.ensemble import RandomForestRegressor

model_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

In [23]:
x_train.shape,y_train.shape

((76451, 19), (76451,))

In [24]:
model_pipeline.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('skewed', ...), ('normal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [25]:
rfg_y_pred = model_pipeline.predict(x_test)

In [26]:
evaluate_regression(y_test,rfg_y_pred)

----- Model Evaluation -----
MAE  : 4.6297
MSE  : 41.8189
RMSE : 6.4668
R2   : 0.3210
MAPE : 57.02%


{'MAE': 4.629673964156957,
 'MSE': 41.818947954438585,
 'RMSE': np.float64(6.466757143610589),
 'R2': 0.321008074129288,
 'MAPE': np.float64(57.0178018290655)}

### 2.XGBoost model

In [27]:
from xgboost import XGBRegressor

xgb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

In [28]:
xgb_pipeline.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('skewed', ...), ('normal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [29]:
xgb_y_pred = xgb_pipeline.predict(x_test)

In [30]:
evaluate_regression(y_test,xgb_y_pred)

----- Model Evaluation -----
MAE  : 4.5828
MSE  : 41.6032
RMSE : 6.4501
R2   : 0.3245
MAPE : 55.68%


{'MAE': 4.582765562369025,
 'MSE': 41.60316316592821,
 'RMSE': np.float64(6.4500514080066225),
 'R2': 0.32451165650739777,
 'MAPE': np.float64(55.67856598085713)}

###  feature shrink

In [ ]:
skewed_colss = [
    # Core logistics
    'delivery_distance',
    'total_product_weight_g',
    'total_volume',
    'freight_ratio',
    'weight_per_item',
    'volume_per_item',

    # Order info
    ##'n_items',
    'total_price',
    'payment_value',
    'payment_installments',

    # Interactions
    'distance_x_weight',
    ## 'logistics_complexity',

    # Seller behavior
    # 'seller_previous_order_count',
    # 'seller_timely_delivery_avg',

    # Structural
    ## 'n_unique_sellers',
    

]
normal_colss = []

categorical_colss = ['category_topfeatures','payment_type']
binary_colss = ['is_same_state']

In [ ]:
x_train_shrink = x_train[skewed_colss + normal_colss + categorical_colss + binary_colss]
x_test_shrink  = x_test[skewed_cols + normal_cols + categorical_cols + binary_cols]

In [ ]:

preprocessor_shrink = ColumnTransformer(transformers=[
    ('skewed', skwed_num_pipeline, skewed_colss),
    ('normal', normal_num_pipeline, normal_colss),
    ('categorical', object_pipeline, categorical_colss),
    ('binary', binary_cols_pipeline, binary_colss)
])

In [ ]:

rfc_pipeline_shrink = Pipeline(steps=[
    ('preprocessing', preprocessor_shrink),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])



In [ ]:
rfc_pipeline_shrink.fit(x_train_shrink,y_train)

In [ ]:
rfc_shrink_y_pred =  rfc_pipeline_shrink.predict(x_test)

In [ ]:
evaluate_regression(y_test,rfc_shrink_y_pred)

In [38]:
# xgboost pipeline shrink


xgb_pipeline_shrink = Pipeline(steps=[
    ('preprocessing', preprocessor_shrink),
    ('model', XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])


In [39]:

xgb_pipeline_shrink.fit(x_train_shrink,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('skewed', ...), ('normal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [40]:
rgb_shrink_y_pred =  xgb_pipeline_shrink.predict(x_test)

In [41]:
evaluate_regression(y_test,rgb_shrink_y_pred)

----- Model Evaluation -----
MAE  : 4.5856
MSE  : 41.5462
RMSE : 6.4456
R2   : 0.3254
MAPE : 55.87%


{'MAE': 4.585583597435661,
 'MSE': 41.546248857809296,
 'RMSE': np.float64(6.445637971357785),
 'R2': 0.325435743735067,
 'MAPE': np.float64(55.866253285644085)}

In [42]:
x_train['n_items'].value_counts()

n_items
1.0     68820
2.0      5870
3.0      1028
4.0       381
5.0       161
6.0       146
7.0        19
8.0         7
10.0        6
12.0        5
11.0        3
9.0         2
13.0        1
20.0        1
21.0        1
Name: count, dtype: int64

In [43]:
x_train['n_unique_sellers'].value_counts()

n_unique_sellers
1.0    75409
2.0      991
3.0       46
4.0        3
5.0        2
Name: count, dtype: int64

In [44]:
x_train['is_same_state'].value_counts()

is_same_state
0    48847
1    27604
Name: count, dtype: int64

In [45]:
x_train.shape

(76451, 19)

In [ ]:
(48847/76451)*100

In [ ]:
(68820/76451)*100